# Ouverture

In [ ]:
# IMPORTS

import sys
from pathlib import Path

import mlflow
import pandas as pd


# PATH
def find_project_directory():
    """
    Détermine le répertoire du projet.
    """
    # Récupérer le chemin d'accès du répertoire courant
    current_dir = Path.cwd()
    # Accéder au répertoire parent (équivalent de os.pardir)
    parent_dir = current_dir.parent
    return str(parent_dir)


%load_ext autoreload
%autoreload 2
# %reload_ext autoreload
project_directory = find_project_directory()
sys.path.append(project_directory)
print("Répertoire ajouté au PATH :", project_directory)

# Imports locaux
from jenedai.ml.models.data_caster_jenedai import DataCaster
from jenedai.ml.models.data_transformer_jenedai import Transformer
from jenedai.ml.models.data_validator_jenedai import DataValidator
from jenedai.ml.models.threshold_selector_jenedai import VarianceThresholdSelector
from jenedai.ml.utils.utils import find_project_directory

# variables
data_folder = "../data"
env_path = Path("../.env")
mlflow.set_tracking_uri("https://jenedai-mlflow.hf.space/")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Répertoire ajouté au PATH : /home/frederic/Documents/JEDHA/projet jenedai/Jenedai


# Code

In [8]:
path_file = Path(data_folder) / "extract_cvs_engis_dataset_500000.csv"

df = pd.read_csv(
    path_file,
    sep=";",
    dtype=str,
)

dataValidator = DataValidator()
df = dataValidator.validate(df)
cast = DataCaster()
df = cast.cast(df)
transformer = Transformer()
df = transformer.transform(df)

[INFO] remove_invalid_rows: 140448 row(s) removed.


## variance theeshold

In [11]:
# Initialize and fit selector
selector = VarianceThresholdSelector(threshold=0.01, normalize=True)
# Quantitative variables
feature_names = [
    "nb_points_soutirage",
    "temperature_2m_mean",
    "relative_humidity_mean",
    "precipitation_sum",
    "total_energie_soutiree_wh",
]
selector.fit(df, feature_names)

print("Selected features : ", selector.selected_features_)

# Display report
print("\nFeature Variance Report:")
print("-" * 60)
report = selector.get_report()
print(report.to_string(index=False))

print(f"\nSelected Features: {len(selector.selected_features_)}")
print(f"Removed Features: {len(selector.removed_features_)}")

# Transform data
df_selected = selector.transform(df)
print(f"\nOriginal shape: {df.shape}")
print(f"Selected shape: {df_selected.shape}")
print(f"\nRemaining features: {df_selected.columns.tolist()}")

DEBUG colonnes numériques : ['nb_points_soutirage', 'temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum', 'total_energie_soutiree_wh']
DEBUG shape : (8680, 5)
DEBUG variances : [nan nan nan nan nan]
DEBUG mask : [False False False False False]
DEBUG selected : []
Selected features :  []

Feature Variance Report:
------------------------------------------------------------
                  Feature  Variance  Selected  Status
      nb_points_soutirage       NaN     False Removed
      temperature_2m_mean       NaN     False Removed
   relative_humidity_mean       NaN     False Removed
        precipitation_sum       NaN     False Removed
total_energie_soutiree_wh       NaN     False Removed

Selected Features: 0
Removed Features: 5

Original shape: (8680, 12)
Selected shape: (8680, 0)

Remaining features: []
